# 🎮 MUKEUS VIDEO ENHANCER — Free Google Colab GPU Processing

Run **MUKEUS VIDEO ENHANCER** virtually on **Google Colab's Free NVIDIA T4 GPU (16 GB VRAM)** with zero load on your local PC!

### Instructions:
1. In Colab top menu, click **Runtime** ➔ **Change runtime type** ➔ Select **T4 GPU** ➔ **Save**.
2. Run **Cell 1** to verify GPU & install dependencies.
3. Run **Cell 2** to launch your live web app!

In [ ]:
# CELL 1: Check GPU & Download Official Cloudflare Binary
!nvidia-smi
import torch
print('='*50)
print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('✅ GPU ACTIVE:', torch.cuda.get_device_name(0))
else:
    print('⚠️ CRITICAL WARNING: Running on CPU mode!')
    print('👉 Please click top menu: Runtime ➔ Change runtime type ➔ Select T4 GPU ➔ Save')
print('='*50)
!apt-get update -qq && apt-get install -y -qq ffmpeg wget
!pip install -q fastapi "uvicorn[standard]" python-multipart pydantic torch torchvision opencv-python numpy imageio-ffmpeg pyngrok nest_asyncio
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /tmp/cloudflared && chmod +x /tmp/cloudflared
print('✅ Official Cloudflare binary installed.')

In [ ]:
# CELL 2: Clone Repo & Launch Application with Confirmed Tunnel Verification
import os, sys, time, subprocess, re

REPO_URL = "https://github.com/mukesh-ram/Mukeus_AI_Video_Enhancer.git"
APP_DIR = "/content/Mukeus_AI_Video_Enhancer"

if not os.path.exists(APP_DIR):
    print(f"📥 Cloning MUKEUS repository from {REPO_URL}...")
    !git clone {REPO_URL} {APP_DIR}
else:
    print("🔄 Pulling latest changes from repository...")
    !git -C {APP_DIR} pull

if APP_DIR not in sys.path:
    sys.path.insert(0, APP_DIR)
os.chdir(APP_DIR)

# 1. Start FastAPI backend server on port 8000
print("⚡ Starting FastAPI Uvicorn Server on port 8000...")
server_process = subprocess.Popen([
    sys.executable, "-m", "uvicorn", "backend.main:app", "--host", "0.0.0.0", "--port", "8000"
])
time.sleep(3)

# 2. Launch official cloudflared tunnel and parse confirmed live URL
print("🌐 Establishing confirmed Cloudflare live tunnel...")
cf_process = subprocess.Popen(
    ["/tmp/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

live_url = None
# Read logs to capture the exact confirmed URL
for i in range(30):
    line = cf_process.stderr.readline()
    if "trycloudflare.com" in line:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            live_url = match.group(0)
            break
    time.sleep(0.5)

print("\n" + "="*65)
if live_url:
    print("✨ YOUR CONFIRMED LIVE MUKEUS WEB APP IS READY AT:")
    print(f"👉 {live_url}")
else:
    print("💡 Direct Localhost URL (Alternative): http://127.0.0.1:8000")
print("="*65 + "\n")

try:
    server_process.wait()
except KeyboardInterrupt:
    server_process.terminate()
    cf_process.terminate()